# From Prompt to Product — Building Your First AI Agent with LangGraph

**Great Learning Masterclass · Participant Workbook**

This notebook builds, in ten numbered steps, the same Research Assistant agent demonstrated
during the session. Each step maps to one slide, so you can re-run the build alone afterwards.

**What you will build**

A question-answering agent that decides for itself whether it needs a tool, calls one or more
tools in an order it chooses, keeps every result in state, and stops when it has enough —
or when it hits the budget you set.

| Step | What you add | Slide |
|---|---|---|
| 1 | A plain LLM call (the baseline) | 16 |
| 2 | Agent state | 16 |
| 3 | The first node | 17 |
| 4 | Edges and the graph | 17 |
| 5 | Tools | 18 |
| 6 | Conditional routing | 18 |
| 7 | The agent loop (back-edge) | 19 |
| 8 | Compile and invoke | 19 |
| 9 | Visualise the graph | 20 |
| 10 | Test it, including the failure cases | 20 |

Run the cells top to bottom. After the API key cell, no further manual intervention is needed.

> **One honest note.** The slides show `builder.add_node("tools", ToolNode(tools))` for brevity.
> `ToolNode` executes tools but does not know about our budget counter, so in Step 5 we wrap it
> in a few lines that also increment `tool_calls_made`. Everything else matches the slides exactly.

create YOUR_GEMINI_API_KEY, use Google AI Studio. 

1. Open Google AI Studio API Keys ( https://aistudio.google.com/apikey?utm_source ) and sign in with your Google account.
2. Click Create API key.
3. Select an existing Google Cloud project, or create a new project when prompted.
4. Google will generate a key. Click Copy.
5. Replace this placeholder:
    

In [1]:
!pip install google-generativeai

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core[grpc] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.4/880.4 kB 16.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple vers

In [2]:
import google.generativeai as genai

# Configure with your API key
genai.configure(api_key="YOUR_API_KEY_HERE")  # Replace with your actual API key

# Create a model instance
model = genai.GenerativeModel("gemini-3.8-flash")

# Generate content
response = model.generate_content("Hello, Gemini!")

print(response.text)


ModuleNotFoundError: No module named 'google'

## 2. Environment setup

First we confirm the Python version. LangGraph needs Python 3.9 or newer; 3.11+ is recommended.

In [1]:
import sys
print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 9), "Please use Python 3.9 or newer."

Python: 3.11.0


## 3. Required packages

We install only three packages: the graph framework, the LangChain core interfaces, and the
OpenAI provider. No vector database, no scraping library — fragile demos teach nothing.

In [2]:
%pip install -q -U langgraph langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 1.11.3 requires jiter<0.11,>=0.6.1, but you have jiter 0.17.0 which is incompatible.
instructor 1.11.3 requires openai<2.0.0,>=1.70.0, but you have openai 3.16.2 which is incompatible.

[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Now we import everything the notebook uses, in one place, so later cells stay short and readable.

In [2]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver

from importlib.metadata import version
for pkg in ("langgraph", "langchain", "langchain-openai"):
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception:
        print(f"{pkg}: not found")

langgraph: 1.2.11
langchain: 1.4.2
langchain-openai: 1.6.2


## 4. API configuration

The key is read from the environment if it is already set, and prompted for otherwise, so the
notebook never stores a credential in a saved cell. This is also the only manual step.

In [3]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

print("Key configured:", bool(os.environ.get("OPENAI_API_KEY")))

Key configured: True


## 5. Step 1 — A plain LLM call

This is the baseline almost everyone starts from: one call in, one string out. `temperature=0`
keeps the model's behaviour repeatable while we are teaching.

Look closely at that answer. It is confident, it is fluent, and it was produced without a
calculator — the model did the arithmetic in its head. That is the gap the rest of this
notebook closes: no state, no tools, and no second step available to it.

## 6. Step 2 — Define the agent state

State is the contract: it declares what is allowed to travel through the graph. The
`add_messages` reducer is what makes `messages` accumulate instead of being overwritten by
each node.

Nothing runs yet — we have only written down the contract. That is deliberate: adding a state
field later is cheap, but discovering that two nodes have been silently overwriting each other
is not.

## 7. Step 3 — The first node

A node is a plain Python function. It receives the whole state and returns **only what changed** —
a diff, which the reducer merges. It never mutates state in place.

## 8. Step 4 — Register the node and wire the edges

`StateGraph` holds the structure. We add the node under a string name, then say where execution
enters (`START`) and leaves (`END`).

This runs — and it is still a chatbot with extra ceremony. The scaffolding only starts paying
for itself in the next two steps, when the model gains tools and the right to choose.

## 9. Step 5 — Create the tools

A tool is a function the model may *request*. The model sees only the name, the docstring and
the type hints, so that text is the interface — write it like documentation, because it is.

Now the first tool. Note how specific the docstring is, including an example of the argument
format — that specificity is what makes the model pick this tool instead of guessing.

The second tool is a small local knowledge base. Keeping it local means the demo cannot be
broken by conference wifi — and a retriever is, architecturally, just another tool.

The third tool stands in for an enterprise system. `FAIL_METRIC_API` is a switch we flip in
Step 10 to watch the agent handle a failing tool without inventing a number.

## 10. Step 5 (continued) — Register the tools with the model

`bind_tools` tells the model which tools exist and what their schemas are. The model can now
*request* a call; it still cannot execute anything.

`ToolNode` executes whichever tool the model asked for and writes the result back as a message.
We wrap it so the same node also increments our budget counter — the one thing the slide
simplified.

The agent node now uses the tool-aware model, so its replies may contain tool requests instead
of a final answer.

## 11. Step 6 — Conditional routing

The router is ordinary Python. The model *requests*; the router *decides*. Every safety control
you will ever add lives in this gap — including the budget check on line five.

When the budget stops a run mid-task, we append an explicit note rather than returning silence.
Saying what it could not finish is the behaviour you want in front of a customer.

## 12. Step 7 — Build the graph and close the loop

The single line `add_edge("tools", "agent")` is the back-edge. It turns a branching workflow into
an agent: reason → act → observe → reason.

## 13. Step 8 — Compile

Compilation validates the graph, wires the reducers, and returns an immutable runnable. The
checkpointer persists every step against a `thread_id`, which is what gives us memory later.

## 14. Step 8 (continued) — Invoke the agent

A small helper keeps the test cells short. Each call gets its own `thread_id` unless we
deliberately reuse one to test memory.

Now the same question we asked in Step 1 — this time the number will be *derived*, not recalled.

## 15. Step 10 — Test case 2: no tool needed

A good agent recognises when it does **not** need to act. Watch the tool-call count stay at zero.

### Test case 3: multi-step reasoning

Two metrics and a comparison — the agent must sequence several tool calls without being told
the order.

## 16. Step 10 — Test case 4: the tool fails

We flip the switch so the metrics API raises. `ToolNode` catches tool exceptions and returns them
as an observation rather than letting them kill the run — we set the flag explicitly here so the
behaviour is visible rather than assumed.

We rebuild the graph with the resilient tools node so the failure path is exercised end to end.

The agent said the lookup failed rather than inventing a figure. Turn the switch back off
before continuing.

In [25]:
FAIL_METRIC_API = False

### Test case 5: the budget ceiling fires

A question that needs more than three tool calls. The agent stops at the limit and says what it
could not finish.

## 17. Optional — Memory and checkpointing

Because the graph was compiled with a checkpointer, reusing a `thread_id` continues the same
conversation. This is what makes a second turn possible at all.

You can also inspect what was persisted. In production this would be a `PostgresSaver`, and the
retention policy would be yours to decide.

## 18. Step 9 — Visualise the graph

The picture is generated from the code, not drawn by hand. If they ever disagree, the code is
right. The PNG renderer needs a network call, so we fall back to Mermaid text.

## 19. Experiments for learners

Each of these takes a few minutes and teaches something the slides only asserted.

**A. Break a docstring.** Change the `calculator` docstring to `"""Does maths stuff."""`,
re-run the registration cell, and ask the FY24 percentage question again. Watch tool selection
degrade. Tool descriptions are prompt engineering.

**B. Move the ceiling.** Set `MAX_TOOL_CALLS = 1` and re-run the multi-step question. Note how
the partial answer changes — and what it costs you in usefulness.

**C. Remove the reducer.** Change `messages` to a plain `list` in `AgentState`, rebuild, and run
any question. The agent forgets mid-run, because each node now overwrites the list.

**D. Add a fourth tool.** Write `get_exchange_rate(pair: str)` returning a fixed dict, register
it, and ask a question that needs it. Notice that you changed nothing else in the graph.

**E. Drop the checkpointer.** Compile without it and re-run the two-turn memory demo. The second
turn loses the context entirely.

Use this cell as scratch space for the experiments above — nothing below depends on it.

In [30]:
# Your experiments here.


## 20. Challenge exercise — add a human approval gate

Extend the agent so that any tool call to a new `issue_refund(order_id: str, amount: float)` tool
pauses for human approval before executing.

**Steps**

1. Write the tool. It should only ever be *requested*, never executed without approval.
2. Add an `approve` node that calls `interrupt("Approve this refund?")` from `langgraph.types`.
3. Change `route` so a requested `issue_refund` goes to `approve` rather than straight to `tools`.
4. Resume the paused run with `graph.invoke(Command(resume="yes"), config)`.

**The point:** the approval gate belongs in the router, between the model's request and its
execution. That is the same gap where budget checks and validation live.

In [ ]:
from langgraph.types import interrupt, Command  # starting imports for the challenge

# Your solution here.


## Key takeaways

1. **State is a contract.** Declare what may travel through the system; reducers decide how
   updates merge. Without `add_messages`, nodes silently overwrite each other.
2. **Nodes are plain functions.** They take the whole state and return only a diff. They never
   mutate in place, which is what keeps a run reconstructable from checkpoints.
3. **Tools give reach; docstrings are the interface.** The model chooses tools almost entirely
   from their names and descriptions.
4. **The model requests, your router decides.** Every safety control — budgets, validation,
   approval gates — lives in that gap.
5. **A loop needs a bound.** A back-edge without a stop condition is a billing incident.
6. **Failure behaviour is designed, not emergent.** The agent reported the API timeout honestly
   because we returned the error as an observation.
7. **Checkpoints give memory and an undo button.** They are also what makes a pause for human
   approval survive a restart.

**Where this goes next:** the notebook is stage one of nine. An API, a UI, real enterprise data
with per-tenant scoping, authentication, observability, evaluation, guardrails and deployment
stand between this and a production agent. The graph is the cheap part.

---
*Great Learning · From Prompt to Product: Building Your First AI Agent with LangGraph*